In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd

ROOT = Path.cwd().parents[1]
kbo = pd.read_csv(ROOT / "data/external/kbo_batting.csv")
mlb = pd.read_csv(ROOT / "data/external/kbo_players_mlb_lines.csv")

def totals(df):
    return df.groupby("player_en")[["pa", "so"]].sum()

k = totals(kbo).add_prefix("kbo_")
m = totals(mlb).add_prefix("mlb_")
d = k.join(m, how="inner").dropna()

MIN_PA = 100
d = d[(d["kbo_pa"] >= MIN_PA) & (d["mlb_pa"] >= MIN_PA)].copy()
d["kbo_k"] = d["kbo_so"] / d["kbo_pa"]
d["mlb_k"] = d["mlb_so"] / d["mlb_pa"]

print(f"{len(d)} players with {MIN_PA}+ PA in both leagues")
print(d[["kbo_pa", "mlb_pa", "kbo_k", "mlb_k"]].describe().round(3).to_string())

47 players with 100+ PA in both leagues
         kbo_pa    mlb_pa   kbo_k   mlb_k
count    47.000    47.000  47.000  47.000
mean    667.660   824.702   0.190   0.257
std     562.725   674.279   0.057   0.060
min     117.000   100.000   0.071   0.122
25%     217.000   238.500   0.144   0.220
50%     503.000   652.000   0.183   0.248
75%    1065.500  1292.500   0.230   0.287
max    2480.000  2937.000   0.298   0.420


In [2]:
import pymc as pm
import arviz as az

# Work in log-odds: rates are bounded [0,1], log-odds are not, so a
# normal hierarchy is appropriate there and shrinkage behaves sensibly
# near the boundaries.
def logit(p):
    return np.log(p / (1 - p))

mlb_logit = logit(d["mlb_k"].values)

with pm.Model() as model:
    # League-level translation: how much log-odds of a strikeout shifts
    # moving from MLB to KBO. Negative means fewer strikeouts in KBO.
    mu = pm.Normal("mu", mu=0.0, sigma=1.0)

    # Between-player spread in that shift. HalfNormal keeps it positive;
    # the data decides whether it is near zero (everyone shifts alike)
    # or large (the shift is personal).
    tau = pm.HalfNormal("tau", sigma=0.5)

    # Each player's own shift, partially pooled toward mu.
    delta = pm.Normal("delta", mu=mu, sigma=tau, shape=len(d))

    kbo_logit = mlb_logit + delta
    kbo_p = pm.math.invlogit(kbo_logit)

    # Observed KBO strikeouts. The binomial carries sample size, so a
    # 100-PA player constrains his own delta far less than a 2,000-PA one.
    pm.Binomial("obs", n=d["kbo_pa"].values, p=kbo_p,
                observed=d["kbo_so"].values)

    idata = pm.sample(2000, tune=2000, target_accept=0.9,
                      random_seed=42, progressbar=True)

print(az.summary(idata, var_names=["mu", "tau"]).round(3).to_string())

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu, tau, delta]


/Users/minjong/Projects/mlb-intelligence-lab/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 1 seconds.


       mean      sd eti89_lb eti89_ub ess_bulk ess_tail r_hat mcse_mean  mcse_sd
mu   -0.427  0.0382    -0.49    -0.37     8252     6253  1.00   0.00042  0.00039
tau  0.2285  0.0321     0.18     0.28     5377     5628  1.00   0.00044  0.00037


In [3]:
delta = idata.posterior["delta"].values.reshape(-1, len(d))
d["delta_post"] = delta.mean(axis=0)
d["delta_sd"] = delta.std(axis=0)

# Raw (no-pooling) estimate: each player's own observed shift
d["delta_raw"] = logit(d["kbo_k"].values) - logit(d["mlb_k"].values)
d["shrink"] = (d["delta_raw"] - d["delta_post"]).abs()

print("shrinkage vs sample size:")
print(d[["kbo_pa", "delta_raw", "delta_post", "shrink"]]
      .sort_values("kbo_pa").head(6).round(3).to_string())
print("  ...")
print(d[["kbo_pa", "delta_raw", "delta_post", "shrink"]]
      .sort_values("kbo_pa").tail(6).round(3).to_string())
print()
print("correlation between KBO PA and shrinkage:",
      round(d["kbo_pa"].corr(d["shrink"]), 3))

shrinkage vs sample size:
                kbo_pa  delta_raw  delta_post  shrink
player_en                                            
Justin Bour        117      0.150      -0.135   0.285
Lewin Diaz         118     -0.408      -0.422   0.014
David Freitas      148     -0.275      -0.349   0.074
Robel Garcia       156     -0.621      -0.548   0.073
Tyler Saladino     163      0.109      -0.110   0.219
Mac Williamson     168      0.053      -0.126   0.180
  ...
                       kbo_pa  delta_raw  delta_post  shrink
player_en                                                   
Jared Hoying             1543     -0.446      -0.446   0.000
Preston Tucker           1569     -0.863      -0.815   0.048
Darin Ruf                1756     -0.546      -0.538   0.008
Socrates Brito           1764     -0.723      -0.698   0.025
Jose Pirela              1856     -0.518      -0.511   0.007
Jose Miguel Fernandez    2480     -0.598      -0.582   0.016

correlation between KBO PA and shrinkage: -0.47

In [5]:
def project_mlb_k(kbo_k, kbo_pa, n_draws=8000, seed=42):
    """Project MLB K% for a KBO player who has never played in MLB.

    Two sources of uncertainty, both carried through:
      1. the league factor mu (well determined, sd 0.038)
      2. this player's personal deviation, drawn from Normal(0, tau)

    The second dominates. Ignoring it would produce intervals far too
    narrow to be honest.
    """
    rng = np.random.default_rng(seed)
    tau_draws = idata.posterior["tau"].values.flatten()
    idx = rng.integers(0, len(mu_draws), n_draws)

    # Observed KBO rate is itself uncertain at small PA — binomial noise
    obs_k = rng.binomial(kbo_pa, kbo_k, n_draws) / kbo_pa
    obs_k = np.clip(obs_k, 1e-4, 1 - 1e-4)

    personal = rng.normal(0, tau_draws[idx])
    mlb_logit_draws = logit(obs_k) - mu_draws[idx] - personal
    return invlogit(mlb_logit_draws)

print("Projected MLB K% for hypothetical KBO hitters:")
print()
for kbo_k, kbo_pa, label in [
    (0.10, 500, "elite contact, full season"),
    (0.15, 500, "good contact, full season"),
    (0.15, 150, "good contact, SMALL sample"),
    (0.22, 500, "average KBO"),
]:
    proj = project_mlb_k(kbo_k, kbo_pa)
    lo, hi = np.percentile(proj, [10, 90])
    print(f"  KBO {kbo_k:.0%} on {kbo_pa} PA  ->  MLB {proj.mean():.1%}  "
          f"[{lo:.1%}, {hi:.1%}]")

Projected MLB K% for hypothetical KBO hitters:



NameError: name 'mu_draws' is not defined

In [6]:
mu_draws = idata.posterior["mu"].values.flatten()

def invlogit(x):
    return 1 / (1 + np.exp(-x))

print("Expected KBO K% for a given MLB K%, with 80% credible interval:")
print()
for mlb_k in [0.15, 0.20, 0.25, 0.30, 0.35]:
    kbo_draws = invlogit(logit(mlb_k) + mu_draws)
    lo, hi = np.percentile(kbo_draws, [10, 90])
    print(f"  MLB {mlb_k:.0%}  ->  KBO {kbo_draws.mean():.1%}  "
          f"[{lo:.1%}, {hi:.1%}]   (ratio {kbo_draws.mean()/mlb_k:.2f})")

Expected KBO K% for a given MLB K%, with 80% credible interval:

  MLB 15%  ->  KBO 10.3%  [9.9%, 10.8%]   (ratio 0.69)
  MLB 20%  ->  KBO 14.0%  [13.5%, 14.6%]   (ratio 0.70)
  MLB 25%  ->  KBO 17.9%  [17.2%, 18.6%]   (ratio 0.71)
  MLB 30%  ->  KBO 21.9%  [21.0%, 22.7%]   (ratio 0.73)
  MLB 35%  ->  KBO 26.0%  [25.1%, 26.9%]   (ratio 0.74)


In [7]:
def project_mlb_k(kbo_k, kbo_pa, n_draws=8000, seed=42):
    """Project MLB K% for a KBO player who has never played in MLB.

    Two sources of uncertainty, both carried through:
      1. the league factor mu (well determined, sd 0.038)
      2. this player's personal deviation, drawn from Normal(0, tau)

    The second dominates. Ignoring it would produce intervals far too
    narrow to be honest.
    """
    rng = np.random.default_rng(seed)
    tau_draws = idata.posterior["tau"].values.flatten()
    idx = rng.integers(0, len(mu_draws), n_draws)

    # Observed KBO rate is itself uncertain at small PA — binomial noise
    obs_k = rng.binomial(kbo_pa, kbo_k, n_draws) / kbo_pa
    obs_k = np.clip(obs_k, 1e-4, 1 - 1e-4)

    personal = rng.normal(0, tau_draws[idx])
    mlb_logit_draws = logit(obs_k) - mu_draws[idx] - personal
    return invlogit(mlb_logit_draws)

print("Projected MLB K% for hypothetical KBO hitters:")
print()
for kbo_k, kbo_pa, label in [
    (0.10, 500, "elite contact, full season"),
    (0.15, 500, "good contact, full season"),
    (0.15, 150, "good contact, SMALL sample"),
    (0.22, 500, "average KBO"),
]:
    proj = project_mlb_k(kbo_k, kbo_pa)
    lo, hi = np.percentile(proj, [10, 90])
    print(f"  KBO {kbo_k:.0%} on {kbo_pa} PA  ->  MLB {proj.mean():.1%}  "
          f"[{lo:.1%}, {hi:.1%}]")

Projected MLB K% for hypothetical KBO hitters:

  KBO 10% on 500 PA  ->  MLB 14.8%  [10.5%, 19.4%]
  KBO 15% on 500 PA  ->  MLB 21.5%  [16.0%, 27.4%]
  KBO 15% on 150 PA  ->  MLB 21.4%  [14.8%, 28.5%]
  KBO 22% on 500 PA  ->  MLB 30.3%  [23.6%, 37.5%]
